In [1]:
import os
from ratelimit import limits, sleep_and_retry
import requests
from urllib3.util import Retry
import sqlite3
import json
from pathlib import Path
import re

In [2]:
start = Path.cwd().resolve()

PROJECT_ROOT = next(
    path
    for path in (start, *start.parents)
    if (path / '.git').exists()
)

DATA_DIRECTORY = PROJECT_ROOT / 'data'

In [3]:
conn = sqlite3.connect(DATA_DIRECTORY / 'league_data.db')
cursor = conn.cursor()

In [4]:
rows = cursor.execute("""
    SELECT
        tables.name AS table_name,
        columns.name AS column_name
    FROM sqlite_schema AS tables
    JOIN pragma_table_info(tables.name) AS columns
    WHERE tables.type = 'table'
      AND tables.name NOT LIKE 'sqlite_%'
    ORDER BY tables.name, columns.cid
""").fetchall()


for table_name, column_name in rows:
    print(table_name, column_name)

bans match_id
bans team_id
bans ban_1
bans ban_2
bans ban_3
bans ban_4
bans ban_5
match_queue match_id
match_queue status
matches match_id
matches team_id
matches top
matches jungle
matches mid
matches bot
matches support
matches patch


In [5]:
EPOCH_TIME_SEPTEMBER2026 = 1788220800
RANKED_SOLO = 420
ONE_SECOND = 1
TWO_MINUTES = 120
NUM_CHAMPIONS_PER_GAME = 10
NUM_GAMES_PER_PLAYER = 1

ROLE_ORDER = ('TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY')

platforms = ['OC1', 'JP1', 'KR', 'BR1', 'LA1', 'LA2', 'NA1', 'TR1', 'RU', 'EUN1', 'EUW1', 'ME1', 'SG2', 'TW2', 'VN2']
regions = {'OC1':'sea', 'SG2': 'sea', 'TW2': 'sea', 'VN2': 'sea', 'JP1': 'asia', 'KR': 'asia', 'BR1': 'americas', 'LA1': 'americas', 'LA2': 'americas', 'NA1': 'americas', 'TR1': 'europe', 'RU': 'europe', 'EUN1': 'europe', 'EUW1': 'europe', 'ME1': 'europe'}

champion_names_url = 'https://ddragon.leagueoflegends.com/cdn/{version}/data/en_US/champion.json'
master_division_url = 'https://{platform}.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'
matches_by_player_url = 'https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?startTime={start_time}&queue={queue}&type=ranked&start=0&count={count}'
match_data_from_matchid_url = 'https://{region}.api.riotgames.com/lol/match/v5/matches/{matchId}'
latest_data_dragon_version_url = 'https://ddragon.leagueoflegends.com/api/versions.json'
api_key = os.getenv("RIOT_API_KEY")

headers = {
    'X-Riot-Token': api_key
}

In [6]:
session = requests.Session()
retries = Retry(total=10,
                backoff_factor=2,
                status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', requests.adapters.HTTPAdapter(max_retries=retries))

In [7]:
@sleep_and_retry
@limits(calls=95, period=TWO_MINUTES)
@limits(calls=18, period=ONE_SECOND)
def call_api(url, headers=None):
    response = session.get(url, headers=headers)

    if response.status_code >= 400:
        print(f'Status: {response.status_code} Url: {url}')
        return None
    
    return response

In [ ]:
# currently up to br1
for platform in platforms[:3]:
    player_data = call_api(master_division_url.format(platform=platform), headers)

    data = player_data.json()['entries']
    player_id = [player['puuid'] for player in data] # collects all the player puuids from the master division

    for puuid in player_id:
        url = matches_by_player_url.format(region=regions[platform],
                                           puuid=puuid,
                                           start_time=EPOCH_TIME_SEPTEMBER2026,
                                           queue=RANKED_SOLO,
                                           count=NUM_GAMES_PER_PLAYER)
        response = call_api(url, headers)
        
        if not response:
            continue

        for match in response.json():
            cursor.execute('INSERT OR IGNORE INTO match_queue (match_id) VALUES (?)', (match,))
            print(f'Added match {match} to queue')

        conn.commit()
        

Added match OC1_709922586 to queue
Added match OC1_709910937 to queue
Added match OC1_709926009 to queue
Added match OC1_709860679 to queue
Added match OC1_709439361 to queue
Added match OC1_709882793 to queue
Added match OC1_709898086 to queue
Added match OC1_709878536 to queue
Added match OC1_709796568 to queue
Added match OC1_709923014 to queue
Added match OC1_709847209 to queue
Added match OC1_709439361 to queue
Added match OC1_709738332 to queue
Added match OC1_709905033 to queue
Added match OC1_709919222 to queue
Added match OC1_709918625 to queue
Added match OC1_709863734 to queue
Added match OC1_709916822 to queue
Added match OC1_709882441 to queue
Added match OC1_709677280 to queue
Added match OC1_709918753 to queue
Added match OC1_709914228 to queue
Added match OC1_709665307 to queue
Added match OC1_709927638 to queue
Added match OC1_709908078 to queue
Added match OC1_709901954 to queue
Added match OC1_709549246 to queue
Added match OC1_709863734 to queue
Added match OC1_7099

In [ ]:
while True:

    cursor.execute('SELECT match_id FROM match_queue WHERE status = "pending" LIMIT 1')
    row = cursor.fetchone()
    
    if not row:
        break

    match_id = row[0]
    platform = match_id.split('_')[0]
    url = match_data_from_matchid_url.format(region=regions[platform],
                                         matchId=match_id)
    
    response = call_api(url, headers)

    if not response:
        cursor.execute('''UPDATE match_queue SET status = 'failed' WHERE match_id = (?)''', (match_id,))
        conn.commit()
        continue

    invalid_roles = False
    
    players = response.json()['info']['participants']

    patch_version_long = response.json()['info']['gameVersion']
    patch_version_short = re.search('[0-9]+.[0-9]+', patch_version_long).group()

    # put each player into a dictionary of dictionary with key team_id -> role
    picks_by_team = {100: {}, 200: {}}
    for player in players:
        role = player['teamPosition']
        team_id = player['teamId']
        picks_by_team[team_id][role] = player['championId']

    # validate that there are indeed 5 distinct roles in each team, fields are sometimes left empty so this is necessary to avoid crashes
    for picks_by_role in picks_by_team.values():
        if set(ROLE_ORDER) != set(picks_by_role.keys()):
            invalid_roles = True
            
    if invalid_roles:
        cursor.execute('''UPDATE match_queue SET status = 'failed' WHERE match_id = (?)''', (match_id,))
        conn.commit()
        continue

    for team_id, picks_by_role in picks_by_team.items():
        ordered_picks = [picks_by_role[role] for role in ROLE_ORDER]

        cursor.execute('''INSERT OR IGNORE INTO matches 
            (match_id, team_id, top, jungle, mid, bot, support, patch)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)''', tuple([match_id] + [team_id] + ordered_picks + [patch_version_short]))


    teams = response.json()['info']['teams']

    for team in teams:
        team_id = team['teamId']

        bans = [ban['championId'] for ban in team['bans']]

        cursor.execute('''INSERT OR IGNORE INTO bans 
                (match_id, team_id, ban_1, ban_2, ban_3, ban_4, ban_5)
                VALUES (?, ?, ?, ?, ?, ?, ?)''', tuple([match_id] + [team_id] + bans))

    cursor.execute('UPDATE match_queue SET status = "processed" WHERE match_id = (?)', (match_id,))
    conn.commit()

cursor.close()
conn.close()

In [ ]:
current_patch = call_api(latest_data_dragon_version_url).json()[0]
# get champion id to names mapping
data = call_api(champion_names_url.format(version=current_patch)).json()['data']
championid_to_name = {data[name]['key']: data[name]['name'] for name in data.keys()}
with open(DATA_DIRECTORY / 'championid_to_name.json', 'w') as f:
    json.dump(championid_to_name, f)
session.close()